In [0]:
%run ./schemas

In [0]:
%run ./common

In [0]:
import json, uuid, traceback
from datetime import date
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set('spark.sql.session.timeZone','UTC')
for widget in ['batch_id','entity_type','repository','run_date','file_path']:
    dbutils.widgets.text(widget,'')
dbutils.widgets.dropdown('load_type','FULL',['FULL','INCREMENTAL'])
batch_id=dbutils.widgets.get('batch_id').strip()
entity=dbutils.widgets.get('entity_type').strip()
repo=dbutils.widgets.get('repository').strip()
run_date=dbutils.widgets.get('run_date').strip()
load_type=dbutils.widgets.get('load_type')
start=now(); execution_id=str(uuid.uuid4()); count=0
try:
    date.fromisoformat(run_date)
    if not batch_id or entity not in ENTITIES or repo not in REPOS:
        raise ValueError('Required batch_id, entity_type, repository, run_date')
    ensure_entity_tables(spark)
    # Select records that the RAW->BRONZE step associates with this batch.
    # On unchanged replay, Bronze intentionally preserves earlier batch_id;
    # hence a strict source file scope is also supported via file_path.
    source_file=dbutils.widgets.get('file_path').strip()
    bronze=(spark.table(f'{CATALOG}.bronze.{entity}')
            .filter(F.col('repository')==repo))
    # Include rows from current batch. Unchanged replays may legitimately select 0.
    bronze=bronze.filter(F.col('batch_id')==batch_id)
    count=bronze.count()
    parsed=bronze.withColumn('api',F.from_json('raw_payload',RAW_SCHEMAS[entity]))
    ts=lambda expr: F.expr(f'try_cast({expr} as timestamp)')

    if entity=='commits':
        silver=parsed.select(
            'record_key','repository',F.col('api.sha').alias('commit_sha'),
            F.col('api.author.id').alias('author_user_id'),
            ts('api.commit.author.date').alias('commit_timestamp'),
            F.col('api.commit.message').alias('message'),
            'batch_id','source_updated_at','payload_hash',
            F.current_timestamp().alias('load_timestamp'))
        reason=F.when(F.col('commit_sha').isNull(),F.lit('NULL_COMMIT_SHA'))
        # Commit date required for daily time-series metrics.
        reason=reason.when(F.col('commit_timestamp').isNull(),F.lit('INVALID_COMMIT_TIMESTAMP'))
    elif entity=='issues':
        silver=parsed.select(
            'record_key','repository',F.col('api.id').alias('issue_id'),
            F.col('api.number').alias('issue_number'),
            F.col('api.user.id').alias('author_user_id'),
            F.col('api.title').alias('title'),F.col('api.state').alias('state'),
            ts('api.created_at').alias('created_at'),
            ts('api.updated_at').alias('updated_at'),
            ts('api.closed_at').alias('closed_at'),
            'batch_id','source_updated_at','payload_hash',
            F.current_timestamp().alias('load_timestamp'))
        reason=(F.when(F.col('issue_id').isNull(),F.lit('NULL_ISSUE_ID'))
                  .when(F.col('created_at').isNull() | F.col('updated_at').isNull(),
                        F.lit('INVALID_TIMESTAMP')))
    else:
        silver=parsed.select(
            'record_key','repository',F.col('api.id').alias('pr_id'),
            F.col('api.number').alias('pr_number'),
            F.col('api.user.id').alias('author_user_id'),
            F.col('api.title').alias('title'),F.col('api.state').alias('state'),
            F.col('api.draft').alias('draft'),
            ts('api.created_at').alias('created_at'),
            ts('api.updated_at').alias('updated_at'),
            ts('api.closed_at').alias('closed_at'),
            ts('api.merged_at').alias('merged_at'),
            'batch_id','source_updated_at','payload_hash',
            F.current_timestamp().alias('load_timestamp'))
        reason=(F.when(F.col('pr_id').isNull(),F.lit('NULL_PR_ID'))
                  .when(F.col('created_at').isNull() | F.col('updated_at').isNull(),
                        F.lit('INVALID_TIMESTAMP')))
    silver=silver.withColumn('_reason',reason)
    failed=silver.filter(F.col('_reason').isNotNull())
    # Join original Bronze payload for forensic quarantine.
    q=(failed.join(bronze.select('record_key','raw_payload'),on='record_key',how='left')
       .select(
          F.sha2(F.concat_ws('|',F.lit(batch_id),F.lit(entity),'record_key','_reason'),256)
             .alias('quarantine_id'),
          F.lit(batch_id).alias('batch_id'),F.col('repository'),F.lit(entity).alias('entity'),
          F.col('record_key'),F.col('raw_payload'),F.col('_reason').alias('error_type'),
          F.concat(F.lit('Silver validation failure: '),'_reason').alias('error_reason'),
          F.current_timestamp().alias('load_timestamp'),
          F.current_timestamp().alias('quarantined_at')))
    quarantined=quarantine(spark,q,'silver')
    valid=silver.filter(F.col('_reason').isNull()).drop('_reason')
    # Deduplicate before MERGE (one source row per target record key).
    rank=Window.partitionBy('record_key').orderBy(F.col('source_updated_at').desc_nulls_last(),
                                                  F.col('payload_hash').desc())
    valid=valid.withColumn('_rn',F.row_number().over(rank)).filter('_rn=1').drop('_rn')
    valid_count=valid.count()
    upserted=merge_changed(spark,valid,f'{CATALOG}.silver.{entity}')
    print(f'SILVER {repo}/{entity}: inspected={count} valid={valid_count} quarantined={quarantined}')
    log(spark,execution_id,batch_id,'bronze_to_silver','SILVER',run_date,load_type,start,
        'SUCCESS',count,f'valid={valid_count}; quarantined={quarantined}',
        upserted=upserted, quarantined=quarantined)
except Exception:
    err=traceback.format_exc()
    try: log(spark,execution_id,batch_id,'bronze_to_silver','SILVER',run_date or '1970-01-01',
             load_type,start,'FAILED',count,err)
    except Exception as e: print('LOGGING FAILED:',e)
    raise

dbutils.notebook.exit(json.dumps({'status':'SUCCESS','rows_processed':count,
                                   'valid_keys':valid_count,'upserted':upserted,'quarantined':quarantined}))
